# Robo-Greeno Hexapod — Turn-in-Place Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KamaTechOrg/robo-greeno-data-a/blob/main/ShiraMarzel/turn_in_place_demo.ipynb)

A self-contained MuJoCo simulation of a six-legged robot **spinning on the
spot**. The body geometry, a closed-form inverse-kinematics solver, the MuJoCo
model and the turn gait are all generated from a single `config.py` — no
external mesh, URDF or dataset download.

The robot reuses the whole alternating-tripod scaffold from the straight walk
and changes only **the path each foot follows**: in stance the foot sweeps
along an *arc* about the body centre instead of sliding straight back. Every
stance foot rotates by `-d`, so the body rotates by `+d` — do it each step and
the turns add up.

**How to run:** open in Google Colab (badge above) and choose
**Runtime → Run all**. Every cell runs top-to-bottom with no edits. It also
runs locally: `pip install mujoco mediapy`, then run all cells.

What you get:
1. The three shared modules + the demo are written to disk.
2. A **headless self-test** proving the gait is reachable and the robot turns
   in place without walking away (must print `ALL CHECKS PASSED`).
3. An **inline video** of the hexapod spinning, rendered offscreen.

## 1 · Install dependencies

In [ ]:
%pip install -q mujoco mediapy

## 2 · Write the hexapod modules

These four cells write the exact source from
[`ShiraMarzel/turn-in-place/`](https://github.com/KamaTechOrg/robo-greeno-data-a/tree/main/ShiraMarzel/turn-in-place)
into the working directory, so the notebook is fully self-contained.

The first three files are **shared, unedited** (the same robot every demo
uses). The fourth, `demo_turn_in_place.py`, is the part Shira owns: the turn.

In [ ]:
%%writefile config.py
"""
config.py  --  the ONE geometry file for the Robo-Greeno hexapod.

Everything about the robot's shape lives here. The MJCF model, the
inverse kinematics and the runner all read these numbers and nothing
else. When the real PhantomX hardware lands, you edit this file only
-- no other file needs to change. That is the whole point.

Units: metres and radians.  Frame: +X forward, +Y left, +Z up.
"""

import math

# --------------------------------------------------------------------
# Leg link lengths  (one 3-DOF leg: coxa -> femur -> tibia)
# Same 4 : 8 : 13 proportions as the Stage A leg explorer, in metres.
# Swap for real AX-12 dimensions when the hardware lands.
# --------------------------------------------------------------------
COXA  = 0.040   # L1  hip link  (horizontal swing link)
FEMUR = 0.080   # L2  thigh
TIBIA = 0.130   # L3  shin

FOOT_RADIUS = 0.012   # rounded foot tip -- also the ground contact radius

# --------------------------------------------------------------------
# Body
# --------------------------------------------------------------------
BODY_RADIUS = 0.100   # centre of body  ->  each coxa joint
BODY_HALF_H = 0.018   # half the trunk thickness
TRUNK_MASS  = 0.45    # kg  (small-scale learning model)

# --------------------------------------------------------------------
# The six legs.  Each is (name, mount angle).
# Mount angle: body XY-plane, 0 = forward, positive = CCW (toward +Y).
# --------------------------------------------------------------------
LEGS = [
    ("front_left",   math.radians(  45.0)),
    ("mid_left",     math.radians(  90.0)),
    ("back_left",    math.radians( 135.0)),
    ("back_right",   math.radians(-135.0)),
    ("mid_right",    math.radians( -90.0)),
    ("front_right",  math.radians( -45.0)),
]

# --------------------------------------------------------------------
# Alternating tripod gait groups (indices into LEGS).
# One tripod is on the ground while the other swings.
# --------------------------------------------------------------------
TRIPOD_A = [0, 2, 4]   # front_left, back_left, mid_right
TRIPOD_B = [1, 3, 5]   # mid_left,  back_right, front_right

# --------------------------------------------------------------------
# Joint travel limits  (radians)
# --------------------------------------------------------------------
COXA_RANGE  = (math.radians(-50.0), math.radians( 50.0))
FEMUR_RANGE = (math.radians(-90.0), math.radians(120.0))
TIBIA_RANGE = (math.radians(-170.0), math.radians( 20.0))

# --------------------------------------------------------------------
# Default standing stance
#   STANCE_RADIUS  -- foot distance from body centre, on the ground
#   STANCE_HEIGHT  -- height of the body centre above the ground
# --------------------------------------------------------------------
STANCE_RADIUS = 0.200
STANCE_HEIGHT = 0.075

# --------------------------------------------------------------------
# Tripod walk parameters (used by run.py --walk)
# --------------------------------------------------------------------
GAIT_PERIOD = 1.4     # seconds for one full A+B cycle
GAIT_STRIDE = 0.060   # metres a foot travels along +X per cycle
GAIT_LIFT   = 0.030   # metres a swinging foot lifts off the ground


def describe():
    """Print a short human summary of the configured robot."""
    span = 2.0 * STANCE_RADIUS
    leg_reach = COXA + FEMUR + TIBIA
    print("Robo-Greeno hexapod -- configured geometry")
    print(f"  legs            : {len(LEGS)}  x 3 DOF = {3 * len(LEGS)} joints")
    print(f"  link lengths    : coxa {COXA*100:.0f} cm | "
          f"femur {FEMUR*100:.0f} cm | tibia {TIBIA*100:.0f} cm")
    print(f"  max leg reach   : {leg_reach*100:.0f} cm")
    print(f"  stance width    : {span*100:.0f} cm  (foot to foot)")
    print(f"  body ride height: {STANCE_HEIGHT*100:.0f} cm")


if __name__ == "__main__":
    describe()


In [ ]:
%%writefile hexapod_ik.py
"""
hexapod_ik.py  --  closed-form kinematics for one 3-DOF hexapod leg.

This is the exact same maths as the Stage A leg explorer
(hexapod-leg-ik-explorer.html), packaged for the MuJoCo runner.
No iteration, no solver -- pure trigonometry students can derive on
paper and check by hand.

A leg has three joints:
  coxa   -- yaw, swings the whole leg left/right
  femur  -- pitch, lifts the leg in its own vertical plane
  tibia  -- pitch at the knee

Joint sign convention (matches the MJCF model):
  coxa  > 0  -> leg swings toward +Y
  femur > 0  -> leg lifts upward
  tibia      -> 0 is a straight leg, negative folds the foot under
"""

import math

import config as cfg


def leg_ik(x, y, z, L1=cfg.COXA, L2=cfg.FEMUR, L3=cfg.TIBIA):
    """Inverse kinematics: foot target -> three joint angles.

    x, y, z : desired foot position relative to the COXA joint, in
              that leg's own frame (+x points out along the leg).
    Returns (coxa, femur, tibia) in radians, or None if out of reach.
    """
    coxa = math.atan2(y, x)                 # top view: aim the leg
    r    = math.hypot(x, y)                 # horizontal run to the foot
    rho  = r - L1                           # reach past the coxa link
    D    = math.hypot(rho, z)               # femur joint -> foot

    # reach test: the femur+tibia 2-link arm must be able to span D
    if not (abs(L2 - L3) - 1e-9 <= D <= L2 + L3 + 1e-9):
        return None

    cos_knee = (L2 * L2 + L3 * L3 - D * D) / (2.0 * L2 * L3)
    knee     = math.acos(_clamp(cos_knee))

    cos_beta = (L2 * L2 + D * D - L3 * L3) / (2.0 * L2 * D)
    beta     = math.acos(_clamp(cos_beta))

    femur = math.atan2(z, rho) + beta       # knee-up solution
    tibia = knee - math.pi                  # 0 = straight leg
    return (coxa, femur, tibia)


def leg_fk(coxa, femur, tibia, L1=cfg.COXA, L2=cfg.FEMUR, L3=cfg.TIBIA):
    """Forward kinematics: three joint angles -> foot (x, y, z).
    Used to verify the IK by round-trip:  FK(IK(target)) == target."""
    pitch_t = femur + tibia                 # absolute pitch of the tibia
    rho_foot = L1 + L2 * math.cos(femur) + L3 * math.cos(pitch_t)
    z_foot = L2 * math.sin(femur) + L3 * math.sin(pitch_t)
    return (rho_foot * math.cos(coxa),
            rho_foot * math.sin(coxa),
            z_foot)


def body_target_to_leg(foot_body, mount_angle, body_radius=cfg.BODY_RADIUS):
    """Body frame -> leg frame.

    A foot target is naturally given in the body frame. Each leg's
    coxa joint sits on the body rim at its mount angle and the leg's
    own +x axis points radially outward. This rotates a body-frame
    target into the leg frame that leg_ik() expects."""
    fx, fy, fz = foot_body
    dx = fx - body_radius * math.cos(mount_angle)
    dy = fy - body_radius * math.sin(mount_angle)
    c, s = math.cos(mount_angle), math.sin(mount_angle)
    return (dx * c + dy * s,
            -dx * s + dy * c,
            fz)


def solve_all(foot_targets_body):
    """Solve every leg at once.

    foot_targets_body : 6 (x, y, z) foot targets in the body frame,
                        one per leg in config.LEGS order.
    Returns 6 (coxa, femur, tibia) tuples.
    Raises ValueError if any leg cannot reach its target."""
    angles = []
    for (name, mount), target in zip(cfg.LEGS, foot_targets_body):
        leg_xyz = body_target_to_leg(target, mount)
        sol = leg_ik(*leg_xyz)
        if sol is None:
            raise ValueError(f"leg '{name}' cannot reach {target}")
        angles.append(sol)
    return angles


def default_stance(stance_radius=None, stance_height=None):
    """Return 6 foot targets in the body frame for a neutral stand.

    The body frame sits at the trunk centre, so a foot resting on the
    ground is FOOT_RADIUS - STANCE_HEIGHT below it (the rounded foot
    tip touches the ground, its centre sits one radius higher)."""
    R = cfg.STANCE_RADIUS if stance_radius is None else stance_radius
    H = cfg.STANCE_HEIGHT if stance_height is None else stance_height
    foot_z = cfg.FOOT_RADIUS - H
    return [(R * math.cos(mount), R * math.sin(mount), foot_z)
            for name, mount in cfg.LEGS]


def _clamp(v, lo=-1.0, hi=1.0):
    return max(lo, min(hi, v))


if __name__ == "__main__":
    print("IK / FK round-trip on the default standing stance\n")
    worst = 0.0
    for (name, mount), target in zip(cfg.LEGS, default_stance()):
        leg_xyz = body_target_to_leg(target, mount)
        sol = leg_ik(*leg_xyz)
        err = math.dist(leg_xyz, leg_fk(*sol))
        worst = max(worst, err)
        deg = tuple(round(math.degrees(a), 1) for a in sol)
        print(f"  {name:12s}  coxa/femur/tibia = {deg}  err = {err:.2e} m")
    print(f"\nworst round-trip error: {worst:.2e} m  "
          f"({'PASS' if worst < 1e-9 else 'FAIL'})")


In [ ]:
%%writefile hexapod_model.py
"""
hexapod_model.py  --  build the MuJoCo model of the hexapod.

The whole robot is generated in code from config.py, so there is no
separate hand-edited XML to keep in sync. Call build_mjcf() to get the
MJCF (MuJoCo's XML) as a string, or run this file directly to write a
hexapod.xml you can inspect.

A self-contained PhantomX-class hexapod: a round trunk with six
identical 3-DOF legs, 18 hinge joints, one position servo per joint,
and a ground plane. No external mesh or URDF download needed.
"""

import math

import config as cfg


def _leg(name, mount):
    """Return the MJCF body block for one leg: coxa -> femur -> tibia."""
    rx = cfg.BODY_RADIUS * math.cos(mount)
    ry = cfg.BODY_RADIUS * math.sin(mount)
    c0, c1 = cfg.COXA_RANGE
    f0, f1 = cfg.FEMUR_RANGE
    t0, t1 = cfg.TIBIA_RANGE
    L1, L2, L3 = cfg.COXA, cfg.FEMUR, cfg.TIBIA
    rf = cfg.FOOT_RADIUS
    return f"""
      <body name="{name}_coxa" pos="{rx:.6f} {ry:.6f} 0" euler="0 0 {mount:.6f}">
        <joint name="{name}_coxa" axis="0 0 1" range="{c0:.6f} {c1:.6f}"/>
        <geom type="capsule" fromto="0 0 0 {L1:.6f} 0 0" size="0.012" rgba="0.60 0.58 0.54 1"/>
        <body name="{name}_femur" pos="{L1:.6f} 0 0">
          <joint name="{name}_femur" axis="0 -1 0" range="{f0:.6f} {f1:.6f}"/>
          <geom type="capsule" fromto="0 0 0 {L2:.6f} 0 0" size="0.010" rgba="0.11 0.62 0.46 1"/>
          <body name="{name}_tibia" pos="{L2:.6f} 0 0">
            <joint name="{name}_tibia" axis="0 -1 0" range="{t0:.6f} {t1:.6f}"/>
            <geom type="capsule" fromto="0 0 0 {L3:.6f} 0 0" size="0.008" rgba="0.18 0.49 0.85 1"/>
            <geom name="{name}_foot" type="sphere" pos="{L3:.6f} 0 0" size="{rf:.6f}" rgba="0.85 0.35 0.19 1"/>
            <site name="{name}_foot" pos="{L3:.6f} 0 0" size="0.006"/>
          </body>
        </body>
      </body>"""


def _actuators():
    rows = []
    for name, _ in cfg.LEGS:
        for joint, rng in (("coxa", cfg.COXA_RANGE),
                           ("femur", cfg.FEMUR_RANGE),
                           ("tibia", cfg.TIBIA_RANGE)):
            kp = 18.0 if joint == "coxa" else 30.0
            rows.append(
                f'    <position name="{name}_{joint}" joint="{name}_{joint}" '
                f'kp="{kp}" ctrlrange="{rng[0]:.6f} {rng[1]:.6f}"/>')
    return "\n".join(rows)


def build_mjcf():
    """Return the complete MuJoCo model as an MJCF (XML) string."""
    legs = "".join(_leg(name, mount) for name, mount in cfg.LEGS)
    spawn_z = cfg.STANCE_HEIGHT
    return f"""<mujoco model="robo_greeno_hexapod">
  <compiler angle="radian" autolimits="true"/>
  <option timestep="0.002" integrator="implicitfast" gravity="0 0 -9.81"/>

  <default>
    <joint damping="0.14" armature="0.012"/>
    <geom friction="1.1 0.06 0.01" density="700"/>
  </default>

  <visual>
    <headlight diffuse="0.5 0.5 0.5" ambient="0.4 0.4 0.4"/>
    <rgba haze="0.95 0.95 0.93 1"/>
  </visual>

  <worldbody>
    <light pos="0 0 1.4" dir="0 0 -1" diffuse="0.7 0.7 0.7"/>
    <geom name="ground" type="plane" size="3 3 0.1" rgba="0.92 0.91 0.87 1"/>

    <body name="trunk" pos="0 0 {spawn_z:.4f}">
      <freejoint name="trunk"/>
      <geom name="trunk" type="cylinder" size="{cfg.BODY_RADIUS:.4f} {cfg.BODY_HALF_H:.4f}"
            mass="{cfg.TRUNK_MASS}" rgba="0.36 0.35 0.33 1"/>
      <site name="trunk_center" pos="0 0 0" size="0.01"/>{legs}
    </body>
  </worldbody>

  <actuator>
{_actuators()}
  </actuator>
</mujoco>
"""


def save(path="hexapod.xml"):
    """Write the generated MJCF to a file and return the path."""
    with open(path, "w") as fh:
        fh.write(build_mjcf())
    return path


if __name__ == "__main__":
    p = save()
    print(f"wrote {p}")
    print(f"  {3 * len(cfg.LEGS)} joints, {3 * len(cfg.LEGS)} position servos, "
          f"{len(cfg.LEGS)} legs")


In [ ]:
%%writefile demo_turn_in_place.py
"""
demo_turn_in_place.py  --  Robo-Greeno hexapod demo: turn in place.

The robot spins on the spot to face a new direction without walking
anywhere. It uses the exact same alternating tripod rhythm as the
straight walk -- swing and stance, tripod A and B, the phase logic --
the one thing that changes is the path each foot follows: a straight
line becomes an arc.

The idea
--------
In the straight walk a stance foot slides backward in a straight line,
which pushes the body forward. To turn instead, a stance foot sweeps
along an *arc* around the body centre. If every stance foot rotates by
a small angle -d about the body's Z axis, the body rotates by +d. Do
that every step and the rotations add up: the robot spins in place.

So we keep each foot's home (hx, hy) and rotate it about the body
centre by an angle a(t):

    x' = hx * cos a  -  hy * sin a
    y' = hx * sin a  +  hy * cos a

In stance the angle sweeps +TURN_ANGLE -> -TURN_ANGLE (foot down,
turning the body); in swing the foot lifts and the angle sweeps back
-TURN_ANGLE -> +TURN_ANGLE to reset for the next push. The vertical
lift dz is exactly the swing lift from the walk demo.

Run it
------
  pip install mujoco
  python demo_turn_in_place.py            # open the 3D viewer
  python demo_turn_in_place.py --check    # headless self-test, no display
  python demo_turn_in_place.py --cw       # turn the other way (clockwise)
"""

import argparse
import math
import sys
import time

import config as cfg
import hexapod_ik as ik
import hexapod_model as model


# --------------------------------------------------------------------
# Turn parameters  --  this is the part you own
# --------------------------------------------------------------------
TURN_ANGLE = math.radians(8.0)   # foot-sweep amplitude per step (try bigger!)
TURN_DIR   = +1.0                # +1 = counter-clockwise (+Z), -1 = clockwise


# --------------------------------------------------------------------
# MuJoCo plumbing -- builds the robot and lets us drive it
# --------------------------------------------------------------------
def make_sim():
    import mujoco
    m = mujoco.MjModel.from_xml_string(model.build_mjcf())
    return mujoco, m, mujoco.MjData(m)


def _aid(mj, m, name):
    return mj.mj_name2id(m, mj.mjtObj.mjOBJ_ACTUATOR, name)


def _jadr(mj, m, name):
    return m.jnt_qposadr[mj.mj_name2id(m, mj.mjtObj.mjOBJ_JOINT, name)]


def _yaw(d, tadr):
    """Trunk heading (rotation about +Z), radians, from its quaternion."""
    qw, qx, qy, qz = (float(d.qpos[tadr + 3]), float(d.qpos[tadr + 4]),
                      float(d.qpos[tadr + 5]), float(d.qpos[tadr + 6]))
    return math.atan2(2.0 * (qw * qz + qx * qy),
                      1.0 - 2.0 * (qy * qy + qz * qz))


def init_stance(mj, m, d):
    """Start the robot already standing so it does not snap on spawn."""
    mj.mj_resetData(m, d)
    t = _jadr(mj, m, "trunk")
    d.qpos[t:t + 7] = [0, 0, cfg.STANCE_HEIGHT, 1, 0, 0, 0]
    for (name, mount), tgt in zip(cfg.LEGS, ik.default_stance()):
        coxa, femur, tibia = ik.leg_ik(*ik.body_target_to_leg(tgt, mount))
        for joint, val in (("coxa", coxa), ("femur", femur), ("tibia", tibia)):
            d.qpos[_jadr(mj, m, f"{name}_{joint}")] = val
    mj.mj_forward(m, d)


def command(mj, m, d, foot_targets):
    """Solve the IK for six foot targets and write the eighteen servos."""
    for (name, _), (coxa, femur, tibia) in zip(cfg.LEGS, ik.solve_all(foot_targets)):
        d.ctrl[_aid(mj, m, f"{name}_coxa")] = coxa
        d.ctrl[_aid(mj, m, f"{name}_femur")] = femur
        d.ctrl[_aid(mj, m, f"{name}_tibia")] = tibia


# --------------------------------------------------------------------
# The demo  --  this is the part you own: the turn gait
# --------------------------------------------------------------------
def turn_targets(t):
    """Foot targets (body frame) for the turn-in-place gait at time t.

    Same tripod scaffold as the straight walk -- only the foot path is
    different: instead of a backward straight line in stance, the foot
    sweeps along an arc about the body centre. The two tripods run the
    same cycle half a period apart, so one tripod is always pushing the
    turn while the other resets, and the robot stays on three feet."""
    base = ik.default_stance()
    out = []
    for i, (name, mount) in enumerate(cfg.LEGS):
        hx, hy, bz = base[i]
        phase = (t / cfg.GAIT_PERIOD) % 1.0
        local = phase if i in cfg.TRIPOD_A else (phase + 0.5) % 1.0
        if local < 0.5:                                   # swing -- reset the foot
            s = local / 0.5
            a = -TURN_ANGLE + s * (2.0 * TURN_ANGLE)
            dz = cfg.GAIT_LIFT * math.sin(math.pi * s)
        else:                                             # stance -- turn the body
            s = (local - 0.5) / 0.5
            a = TURN_ANGLE - s * (2.0 * TURN_ANGLE)
            dz = 0.0
        a *= TURN_DIR
        ca, sa = math.cos(a), math.sin(a)                 # rotate home about +Z
        rx = hx * ca - hy * sa
        ry = hx * sa + hy * ca
        out.append((rx, ry, bz + dz))
    return out


# --------------------------------------------------------------------
# Viewer
# --------------------------------------------------------------------
def view():
    import mujoco
    import mujoco.viewer
    mj, m, d = make_sim()
    init_stance(mj, m, d)
    way = "counter-clockwise" if TURN_DIR > 0 else "clockwise"
    print(f"turn-in-place demo  --  spinning {way}; drag to orbit, close to quit")
    print("  tripod A: front-left, back-left, mid-right")
    print("  tripod B: mid-left, back-right, front-right")
    with mujoco.viewer.launch_passive(m, d) as v:
        start = time.time()
        while v.is_running():
            t = time.time() - start
            command(mj, m, d, turn_targets(t))
            mj.mj_step(m, d)
            v.sync()
            wait = m.opt.timestep - (time.time() - start - t)
            if wait > 0:
                time.sleep(wait)


# --------------------------------------------------------------------
# Headless self-test
# --------------------------------------------------------------------
def check():
    print("turn-in-place demo  --  self-test\n")
    mj, m, d = make_sim()
    print(f"[1] model loaded: {m.nu} servos, {m.nbody} bodies")

    print("[2] every step of the gait is reachable")
    bad = 0
    for k in range(160):                      # 8 s of gait, every 0.05 s
        try:
            ik.solve_all(turn_targets(k * 0.05))
        except ValueError:
            bad += 1
    steps_ok = bad == 0
    print(f"    unreachable steps: {bad}  -> {'PASS' if steps_ok else 'FAIL'}")

    print("[3] the robot turns in place -- it spins but does not walk away")
    init_stance(mj, m, d)
    tadr = _jadr(mj, m, "trunk")
    yaw0 = _yaw(d, tadr)
    for _ in range(4000):                      # 8 s
        command(mj, m, d, turn_targets(d.time))
        mj.mj_step(m, d)
    height = float(d.qpos[tadr + 2])
    x, y = float(d.qpos[tadr + 0]), float(d.qpos[tadr + 1])
    drift = math.hypot(x, y)
    turned = abs((_yaw(d, tadr) - yaw0 + math.pi) % (2 * math.pi) - math.pi)
    upright = not math.isnan(height) and height > 0.045
    spun = math.degrees(turned) > 8.0         # a clear, visible turn
    in_place = drift < 0.05                    # barely moved (< 5 cm)
    print(f"    after 8 s: turned {math.degrees(turned):+.1f} deg, "
          f"drift {drift*100:.1f} cm, ride height {height*100:.1f} cm")
    print(f"    spun {'PASS' if spun else 'FAIL'} | "
          f"in-place {'PASS' if in_place else 'FAIL'} | "
          f"upright {'PASS' if upright else 'FAIL'}")

    ok = steps_ok and spun and in_place and upright
    print(f"\n{'ALL CHECKS PASSED' if ok else 'SOME CHECKS FAILED'}")
    return 0 if ok else 1


def main():
    ap = argparse.ArgumentParser(description="hexapod turn-in-place demo")
    ap.add_argument("--check", action="store_true", help="headless self-test")
    ap.add_argument("--cw", action="store_true", help="turn clockwise instead")
    args = ap.parse_args()
    global TURN_DIR
    if args.cw:
        TURN_DIR = -1.0
    if args.check:
        return check()
    try:
        view()
    except ImportError:
        print("MuJoCo is not installed.  Run:  pip install mujoco")
        return 1
    except Exception as exc:
        print(f"could not open the viewer ({exc}).")
        print("Try the self-test instead:  python demo_turn_in_place.py --check")
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())


## 3 · Configure headless rendering & import

On Colab (and any headless Linux) MuJoCo needs the **EGL** GL backend for
offscreen rendering. We set it *before* importing `mujoco`. Locally on
macOS/Windows the default backend is used automatically.

In [ ]:
import os, sys

# Use EGL for offscreen rendering on Colab / headless Linux.
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG") \
        or (sys.platform.startswith("linux") and not os.environ.get("DISPLAY")):
    os.environ["MUJOCO_GL"] = "egl"

import mujoco
import numpy as np
import demo_turn_in_place as demo      # imports config, hexapod_ik, hexapod_model

print("mujoco", mujoco.__version__, "| GL backend:", os.environ.get("MUJOCO_GL", "default"))

## 4 · Headless self-test

Loads the model, checks every step of the gait is reachable by the IK, then
simulates 8 s and confirms the robot **turned** in place (a clear heading
change) while its `(x, y)` barely moved and it stayed upright.
Must print **`ALL CHECKS PASSED`**.

In [ ]:
demo.check()

## 5 · Watch it spin

Render the simulation offscreen with a high, near-top-down camera so the
rotation is obvious, then show it as an inline video.

In [ ]:
import mediapy as media

mj, m, d = demo.make_sim()
demo.init_stance(mj, m, d)

renderer = mujoco.Renderer(m, height=480, width=640)

cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_TRACKING
cam.trackbodyid = mj.mj_name2id(m, mj.mjtObj.mjOBJ_BODY, "trunk")
cam.distance, cam.azimuth, cam.elevation = 0.95, 90, -65   # look down so the spin shows

DURATION, FPS = 8.0, 30           # seconds of gait, video frame rate
frames = []
while d.time < DURATION:
    demo.command(mj, m, d, demo.turn_targets(d.time))
    mj.mj_step(m, d)
    if len(frames) < d.time * FPS:
        renderer.update_scene(d, camera=cam)
        frames.append(renderer.render())

print(f"rendered {len(frames)} frames")
media.show_video(frames, fps=FPS)

## 6 · Make it yours

The turn gait lives in `turn_targets()` inside `demo_turn_in_place.py`; the
robot's shape lives in `config.py`. To change behaviour, edit the value, re-run
that `%%writefile` cell, then re-run the import / self-test / video cells:

- **Turn faster** — raise `TURN_ANGLE` in `demo_turn_in_place.py`. Find the
  largest angle that still works: past a point a leg cannot reach and
  `solve_all` raises. That angle is the robot's turn-rate limit.
- **Turn the other way** — set `TURN_DIR = -1.0` (or run the script with `--cw`).
- **Harder** — blend the turn with the straight walk so the robot follows a
  **curved path**, or turn to a **target heading** and stop.